### 灵感项目：用 LLM + 工具「挑重要信息」，而不是把整段对话历史原样塞进上下文

## 练习目标（理念）

很多聊天应用会把**全部历史**拼进 `messages`，又贵又容易超上下文窗口。  
本笔记本演示另一种思路：让模型通过 **Tool Calling** 决定何时把对话写入 / 读出 **SQLite**，实现「可恢复的记忆」，而不是每次手动复制粘贴历史。

## 和本课第 2 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Tool Use / Function Calling | `write_history_todb` / `read_history_from_db` |
| System Prompt 约束工具行为 | 要求开场加载、答后存储 |
| Gradio ChatInterface | 浏览器里多轮问答 |
| 外部后端（OpenRouter） | `base_url` 指向 OpenRouter，一套 OpenAI SDK 调多模型 |


### 演示重点

- **工具调用（Tool Calling）**：模型发 `tool_calls`，Python 真正读写数据库，再把结果以 `role=tool` 回传
- **上下文恢复（Context Recovery）**：新会话也能从 SQLite 拉回过往 user/assistant 消息


In [71]:
# ========== 导入：API 客户端、环境变量、UI、数据库、JSON ==========

# 从 openai 导入 OpenAI：兼容 OpenAI 协议的聊天客户端（这里实际连 OpenRouter）
from openai import OpenAI
# 导入标准库 os：用 getenv 读取 OPENAI_API_KEY 等环境变量
import os
# 导入 gradio：快速搭建 ChatInterface 网页聊天 UI
import gradio as gr
# 导入标准库 sqlite3：本地文件型数据库，持久化对话
import sqlite3
# 导入标准库 json：解析 tool arguments，以及把历史列表 dumps 成字符串给模型
import json


In [72]:
# ========== 客户端：走 OpenRouter，而不是直连 api.openai.com ==========

# api_key 从环境变量读取；base_url 指向 OpenRouter 的 OpenAI 兼容端点（URL 保持原样）
openai= OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1",  
)


In [73]:
# ========== 模型 id：聊天用哪个、备用 Claude 名字（字符串必须保持原样）==========

# 主对话模型（OpenRouter 上的 gpt-4.1-mini）
gpt_model = "gpt-4.1-mini"
# 备用/对照用的 Claude 模型 id（本笔记本后续 chat 主要用 gpt_model）
claude_model = "anthropic/claude-3.5-haiku"


### System Prompt 与数据库职责

在系统提示里写清楚：LLM **应该**调用工具，从数据库**读取**旧消息，并在回答后**写入**新的一对 user/assistant 消息。  
下面的英文 `system_prompt` 是发给模型的可运行指令，不要翻译正文。


In [74]:
# ========== System Prompt：强制「先读库、后写库」的工具使用纪律 ==========

# 三引号字符串保持英文：改译会改变模型是否调用工具、调用时机
system_prompt = """
You are a helpful assistant that explains code snippets.

At the start of every conversation you MUST call a tool to load past messages.
After generating a response, you MUST call a tool to store:
- assistant_message
- user_message
"""


In [75]:
# ========== 建库：SQLite 文件 + conversations 表 ==========

# 数据库文件名（相对当前工作目录）；字符串保持原样
DB = "convo.db"

# 连接（with 结束会关闭）；若表不存在则创建
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    # CREATE TABLE IF NOT EXISTS：幂等建表，重复运行笔记本也不会报「表已存在」
    cursor.execute('CREATE TABLE IF NOT EXISTS conversations (id INTEGER PRIMARY KEY AUTOINCREMENT, created_at DATETIME DEFAULT CURRENT_TIMESTAMP, user_message TEXT, assistant_message TEXT)')
    # 提交 DDL/事务
    conn.commit()


In [76]:
# ========== 两个「真工具」函数：写历史 / 读历史 ==========

def write_history_todb(assistant_message, user_message):
    # 打印便于在笔记本里观察「模型是否真的触发了写库工具」
    print("DATABASE TOOL CALLED: Writing history to db", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # 参数化 INSERT（? 占位），避免字符串拼接带来的注入风险
        cursor.execute(
            "INSERT INTO conversations (user_message, assistant_message) VALUES (?, ?)",
            (user_message, assistant_message),
        )
        conn.commit()


def read_history_from_db():
    # 同样打印，方便对照「读库工具是否被调用」
    print("DATABASE TOOL CALLED: Reading history from db", flush=True)
    messages = []

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # 按表中顺序取出每一对 user/assistant 文本
        cursor.execute("SELECT user_message, assistant_message FROM conversations")
        rows = cursor.fetchall()

        # 空库：返回空列表，表示没有可恢复上下文
        if not rows:
            return []  # empty DB case

        # 拼成 Chat Completions 需要的 messages 列表格式
        for user_message, assistant_message in rows:
            messages.append({"role": "user", "content": user_message})
            messages.append({"role": "assistant", "content": assistant_message})

        return messages


In [77]:
# ========== Tool 分发器：根据 tool_call.function.name 调用对应 Python 函数 ==========

def handle_tool_calls(message):
    responses = []
    # 一条助手消息里可能带多个 tool_calls，逐个处理
    for tool_call in message.tool_calls:
        if tool_call.function.name == "write_history_todb":
            # 解析模型传来的 JSON 参数
            arguments = json.loads(tool_call.function.arguments)
            write_history_todb(
                arguments["assistant_message"], arguments["user_message"]
            )
            # 每条 tool 回执必须带上对应的 tool_call_id，模型才能对齐
            responses.append(
                {
                    "role": "tool",
                    "content": "History written to db",
                    "tool_call_id": tool_call.id,
                }
            )
        elif tool_call.function.name == "read_history_from_db":
            # 读库，并把历史列表序列化成 JSON 字符串放进 content（工具结果必须是文本）
            history_messages = read_history_from_db()
            responses.append(
                {
                    "role": "tool",
                    "content": json.dumps(history_messages),
                    "tool_call_id": tool_call.id,
                }
            )
    return responses


In [78]:
# ========== tools 列表：把两个函数的 JSON Schema 交给 Chat Completions ==========

tools = [
    {
        "type": "function",
        "function": {
            "name": "write_history_todb",
            # description 给模型看，保留英文原样
            "description": "Write the conversation history to the database",
            "parameters": {
                "type": "object",
                "properties": {
                    "assistant_message": {"type": "string"},
                    "user_message": {"type": "string"},
                },
                "required": ["assistant_message", "user_message"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_history_from_db",
            "description": "Read the conversation history from the database",
            # 无参数工具：properties 空、required 空列表
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    }
]


In [79]:
# ========== chat：拼 messages → 调模型 → 若有 tool_calls 则循环执行 ==========

def chat(message, history):
    # 从数据库加载历史记录（本实现直接读库；system 仍要求模型也会调读库工具）
    history = read_history_from_db()
    # system + 恢复的历史 + 当前用户新消息
    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )
    # 第一次补全：带上 tools，允许模型发起 function call
    response = openai.chat.completions.create(
        model=gpt_model, messages=messages, tools=tools
    )

    # 标准 tool loop：执行工具 → 把 assistant tool_calls 消息和 tool 结果追加 → 再请求
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(
            model=gpt_model, messages=messages, tools=tools
        )

    # 循环结束后，返回最终助手文本给 Gradio
    return response.choices[0].message.content


In [80]:
# ========== 启动 Gradio ChatInterface（messages 格式）==========

# fn=chat：每条用户输入调用上面的 chat；type="messages" 使用角色消息列表
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Writing history to db
DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Reading history from db
DATABASE TOOL CALLED: Writing history to db
DATABASE TOOL CALLED: Writing history to db
